In [35]:
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split


con = duckdb.connect(database='data/nba.sqlite', read_only=False)
teams = con.query('select * from team').fetchdf()
game_info_df = con.query("select * from game").fetchdf()



In [37]:
team_ids = teams['id']

regular_games = (game_info_df['season_type'] == 'Regular Season') | (game_info_df['season_type'] == 'Playoffs')
game_info_df = game_info_df[regular_games]
game_info_df = game_info_df.dropna(subset="fga_home")
game_info_df['season_id'] = game_info_df['season_id'].astype(int)
game_info_df['year'] = game_info_df['season_id'] % 20000
game_info_df = game_info_df[game_info_df['year'] >= 2000]
game_info_df

,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type,year
36041,22000,1610612742,DAL,Dallas Mavericks,0020000007,2000-10-31,DAL vs. MIL,W,240,35.0,...,16.0,6.0,7.0,20.0,27.0,93.0,-4,0,Regular Season,2000
36042,22000,1610612737,ATL,Atlanta Hawks,0020000004,2000-10-31,ATL vs. CHH,L,240,30.0,...,16.0,7.0,6.0,17.0,23.0,106.0,24,0,Regular Season,2000
36043,22000,1610612741,CHI,Chicago Bulls,0020000006,2000-10-31,CHI vs. SAC,L,240,26.0,...,29.0,12.0,6.0,18.0,23.0,100.0,19,0,Regular Season,2000
36044,22000,1610612761,TOR,Toronto Raptors,0020000005,2000-10-31,TOR vs. DET,L,240,35.0,...,21.0,7.0,6.0,12.0,27.0,104.0,9,0,Regular Season,2000
36045,22000,1610612753,ORL,Orlando Magic,0020000003,2000-10-31,ORL vs. WAS,W,240,34.0,...,20.0,6.0,1.0,27.0,28.0,86.0,-11,0,Regular Season,2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65691,42022,1610612743,DEN,Denver Nuggets,0042200401,2023-06-01,DEN vs. MIA,W,240,40.0,...,26.0,5.0,4.0,8.0,15.0,93.0,-11,1,Playoffs,2022
65692,42022,1610612743,DEN,Denver Nuggets,0042200402,2023-06-04,DEN vs. MIA,L,240,39.0,...,28.0,5.0,4.0,11.0,22.0,111.0,3,1,Playoffs,2022
65693,42022,1610612748,MIA,Miami Heat,0042200403,2023-06-07,MIA vs. DEN,L,240,34.0,...,28.0,3.0,5.0,14.0,18.0,109.0,15,1,Playoffs,2022
65694,42022,1610612748,MIA,Miami Heat,0042200404,2023-06-09,MIA vs. DEN,L,240,35.0,...,26.0,11.0,7.0,8.0,18.0,108.0,13,1,Playoffs,2022


In [38]:
game_info_df.columns

Index(['season_id', 'team_id_home', 'team_abbreviation_home', 'team_name_home',
       'game_id', 'game_date', 'matchup_home', 'wl_home', 'min', 'fgm_home',
       'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home',
       'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home',
       'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home',
       'pts_home', 'plus_minus_home', 'video_available_home', 'team_id_away',
       'team_abbreviation_away', 'team_name_away', 'matchup_away', 'wl_away',
       'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away',
       'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away',
       'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away',
       'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away',
       'season_type', 'year'],
      dtype='object')

In [52]:
train_cols_base = ["fg_pct", "fg3_pct", "ft_pct", "oreb", "dreb", "ast", "stl", "blk", "tov", "pf"]
train_cols = [f"{stat}_home" for stat in train_cols_base] + [f"{stat}_away" for stat in train_cols_base]
test_cols = ["plus_minus_home"]
X = game_info_df[train_cols]
y = game_info_df[test_cols]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

names = ['X_train', 'X_test', 'y_train', 'y_test' ]
vals = [X_train, X_test, y_train, y_test]

for x, n in zip(vals, names):
    x.to_csv(f"data/general_pct_non_possession__{n}.csv")


In [46]:
game_info_df['plus_minus_away'].std()

13.41260236569058